# Auditing and Compliance

## Overview
Healthcare and financial databases operate under regulatory requirements (HIPAA, PIPEDA, SOC 2) that mandate audit trails, access logging, data retention, and privacy-by-design. This notebook covers the practical SQL patterns for compliant audit infrastructure.

| Requirement | Regulation | SQL control |
|---|---|---|
| Access logging | HIPAA §164.312(b) | pgaudit + trigger-based audit_log |
| Minimum necessary | HIPAA §164.502(b) | RLS sensitivity filter + de-identified views |
| Data integrity | SOC 2 CC8 | FK constraints, NOT NULL, checksums |
| Transmission security | HIPAA §164.312(e) | ssl=on, sslmode=verify-full |
| PHI de-identification | PIPEDA, GDPR | Data masking views, tokenisation |

**Dataset:** audit_log pre-populated with 7 events — access grants, a failed login from an external IP, a denied cross-patient access attempt, an export, and a delete.

---

In [ ]:
import sqlite3, hashlib, secrets, base64, hmac
from datetime import datetime

conn = sqlite3.connect(':memory:')
conn.row_factory = sqlite3.Row

conn.executescript("""
CREATE TABLE users (
    user_id TEXT PRIMARY KEY, username TEXT NOT NULL UNIQUE,
    role_name TEXT NOT NULL, department TEXT, active INTEGER DEFAULT 1
);
CREATE TABLE patients (
    patient_id TEXT PRIMARY KEY, full_name TEXT NOT NULL,
    dob TEXT NOT NULL, province TEXT NOT NULL, assigned_clinician TEXT
);
CREATE TABLE patient_records (
    record_id TEXT PRIMARY KEY, patient_id TEXT NOT NULL,
    record_type TEXT NOT NULL, content TEXT NOT NULL,
    sensitivity TEXT DEFAULT 'normal', created_by TEXT NOT NULL, created_at TEXT
);
CREATE TABLE financial_accounts (
    account_id TEXT PRIMARY KEY, customer_id TEXT NOT NULL,
    account_type TEXT NOT NULL, balance_enc TEXT, ssn_hash TEXT, created_at TEXT
);
CREATE TABLE audit_log (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT, event_time TEXT DEFAULT (datetime('now')),
    username TEXT NOT NULL, action TEXT NOT NULL, table_name TEXT,
    record_id TEXT, old_value TEXT, new_value TEXT, ip_address TEXT, success INTEGER DEFAULT 1
);
CREATE TABLE role_permissions (
    role_name TEXT NOT NULL, table_name TEXT NOT NULL,
    can_select INTEGER DEFAULT 0, can_insert INTEGER DEFAULT 0,
    can_update INTEGER DEFAULT 0, can_delete INTEGER DEFAULT 0,
    row_filter TEXT, PRIMARY KEY (role_name, table_name)
);
""")

conn.executemany("INSERT INTO users VALUES (?,?,?,?,?)", [
    ('U001','dr.lee',    'clinician','Cardiology',    1),
    ('U002','dr.pham',   'clinician','Endocrinology',1),
    ('U003','analyst1',  'analyst',  'Reporting',    1),
    ('U004','admin1',    'admin',    'IT',           1),
    ('U005','auditor1',  'auditor',  'Compliance',   1),
    ('U006','patient_p001','patient',None,            1),
])
conn.executemany("INSERT INTO patients VALUES (?,?,?,?,?)", [
    ('P001','Alice Ngata',  '1980-03-15','NB','dr.lee'),
    ('P002','Bob Chen',     '1972-07-22','ON','dr.pham'),
    ('P003','Fatima Rashid','1986-11-05','BC','dr.lee'),
    ('P004','James MacLeod','1963-01-30','NS','dr.pham'),
    ('P005','Mei-Ling Tran','1966-08-19','QC','dr.lee'),
])
conn.executemany("INSERT INTO patient_records VALUES (?,?,?,?,?,?,?)", [
    ('R001','P001','diagnosis',  'Hypertension, Stage 2',      'normal',    'dr.lee', '2024-01-15'),
    ('R002','P001','prescription','Lisinopril 10mg daily',     'normal',    'dr.lee', '2024-01-15'),
    ('R003','P002','diagnosis',  'Type 2 Diabetes',            'normal',    'dr.pham','2024-01-10'),
    ('R004','P002','lab',        'HbA1c 8.4%',                 'normal',    'dr.pham','2024-01-10'),
    ('R005','P003','diagnosis',  'Asthma, moderate persistent','sensitive', 'dr.lee', '2024-02-01'),
    ('R006','P004','diagnosis',  'CKD Stage 3, T2DM',          'sensitive', 'dr.pham','2024-02-15'),
    ('R007','P005','prescription','Insulin glargine 20u',      'restricted','dr.lee', '2024-03-01'),
])
salt = secrets.token_hex(8)
def hash_ssn(ssn): return hashlib.sha256((salt+ssn).encode()).hexdigest()
conn.executemany("INSERT INTO financial_accounts VALUES (?,?,?,?,?,datetime('now'))", [
    ('ACC001','P001','chequing','[encrypted:AES256]',hash_ssn('123-45-6789')),
    ('ACC002','P002','savings', '[encrypted:AES256]',hash_ssn('987-65-4321')),
    ('ACC003','P003','chequing','[encrypted:AES256]',hash_ssn('456-78-9012')),
])
conn.executemany("INSERT INTO role_permissions VALUES (?,?,?,?,?,?,?)", [
    ('clinician','patients',       1,0,1,0,'assigned_clinician = current_user'),
    ('clinician','patient_records',1,1,1,0,'patient_id IN (SELECT patient_id FROM patients WHERE assigned_clinician = current_user)'),
    ('analyst',  'patients',       1,0,0,0,'province IS NOT NULL'),
    ('analyst',  'patient_records',1,0,0,0,"sensitivity = 'normal'"),
    ('admin',    'patients',       1,1,1,1,None),
    ('admin',    'patient_records',1,1,1,1,None),
    ('auditor',  'audit_log',      1,0,0,0,None),
    ('auditor',  'patients',       1,0,0,0,None),
    ('patient',  'patients',       1,0,0,0,'patient_id = current_patient_id()'),
    ('patient',  'patient_records',1,0,0,0,'patient_id = current_patient_id()'),
])
conn.executemany(
    "INSERT INTO audit_log (username,action,table_name,record_id,old_value,new_value,ip_address,success) VALUES (?,?,?,?,?,?,?,?)", [
    ('dr.lee',  'SELECT','patient_records','R001',None,None,'10.0.1.5',1),
    ('dr.lee',  'UPDATE','patient_records','R002','{"content":"Lisinopril 5mg"}','{"content":"Lisinopril 10mg"}','10.0.1.5',1),
    ('analyst1','SELECT','patients',       None,  None,None,'10.0.2.8',1),
    ('analyst1','EXPORT','patient_records',None,  None,None,'10.0.2.8',1),
    ('unknown', 'LOGIN', None,             None,  None,None,'203.0.113.9',0),
    ('dr.pham', 'SELECT','patient_records','R005',None,None,'10.0.1.6',0),
    ('admin1',  'DELETE','patient_records','R007',None,None,'10.0.1.1',1),
])
conn.commit()
print("Dataset loaded:")
for t in ['users','patients','patient_records','financial_accounts','audit_log','role_permissions']:
    print(f"  {t}: {conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]} rows")

print("=== Audit log: contents and analysis ===")
rows = conn.execute("""
    SELECT log_id, event_time, username, action,
           COALESCE(table_name,'—') AS tbl,
           COALESCE(record_id,'—') AS rec,
           success, ip_address
    FROM audit_log ORDER BY log_id
""").fetchall()
print(f"  {'id':<4s}  {'username':<12s}  {'action':<8s}  {'table':<16s}  {'record':<8s}  {'ok':<4s}  ip")
print("  " + "-"*70)
for r in rows:
    ok = '✓' if r['success'] else '✗ DENIED'
    print(f"  {r['log_id']:<4d}  {r['username']:<12s}  {r['action']:<8s}  {r['tbl']:<16s}  {r['rec']:<8s}  {ok:<8s}  {r['ip_address']}")

print()
print("Failed / denied access attempts:")
failed = conn.execute("""
    SELECT username, COUNT(*) AS n, GROUP_CONCAT(ip_address) AS ips
    FROM audit_log WHERE success=0 GROUP BY username
""").fetchall()
for r in failed:
    print(f"  {r[0]}: {r[1]} failure(s) from {r[2]}")

print()
print("PostgreSQL audit trigger:")
print("""
  CREATE OR REPLACE FUNCTION audit_trigger() RETURNS trigger AS $$
  BEGIN
      INSERT INTO audit_log (username, action, table_name, record_id, old_value, new_value)
      VALUES (
          current_user, TG_OP, TG_TABLE_NAME,
          COALESCE(NEW.record_id, OLD.record_id),
          CASE TG_OP WHEN 'DELETE' THEN row_to_json(OLD)::text
                     WHEN 'UPDATE' THEN row_to_json(OLD)::text ELSE NULL END,
          CASE TG_OP WHEN 'INSERT' THEN row_to_json(NEW)::text
                     WHEN 'UPDATE' THEN row_to_json(NEW)::text ELSE NULL END
      );
      RETURN COALESCE(NEW, OLD);
  END;
  $$ LANGUAGE plpgsql SECURITY DEFINER;

  CREATE TRIGGER audit_patient_records
      AFTER INSERT OR UPDATE OR DELETE ON patient_records
      FOR EACH ROW EXECUTE FUNCTION audit_trigger();
""")


In [ ]:

import hashlib
print("=== Data masking for non-production environments ===")
print()
print("Never copy production PHI/PII to development or test environments.")
print()

def mask_name(name): return ' '.join(p[0]+'***' for p in name.split())
def mask_dob(dob):   return dob[:4]+'-01-01'
def tokenise_id(pid): return 'T'+hashlib.sha256(('dev-salt'+pid).encode()).hexdigest()[:8].upper()

patients = conn.execute("SELECT patient_id,full_name,dob,province FROM patients").fetchall()
print("Masked patients table (safe for dev/test):")
print(f"  {'token_id':<10s}  {'masked_name':<14s}  {'birth_year':<12s}  province")
print("  " + "-"*50)
for p in patients:
    print(f"  {tokenise_id(p['patient_id']):<10s}  {mask_name(p['full_name']):<14s}  {mask_dob(p['dob']):<12s}  {p['province']}")

print()
print("PostgreSQL masked view DDL:")
print("""
  CREATE VIEW analytics.patients_masked AS
  SELECT
      encode(digest(patient_id || 'secret_seed', 'sha256'), 'hex') AS patient_token,
      LEFT(full_name, 1) || '***'   AS name_masked,
      EXTRACT(YEAR FROM dob::date)  AS birth_year,
      province
  FROM patients;

  -- Synthetic data for test DB:
  INSERT INTO patients_test
  SELECT
      'P' || LPAD(seq::text, 3, '0') AS patient_id,
      (ARRAY['Alice','Bob','Carol'])[1+(random()*2)::int] || ' Test' AS full_name,
      (DATE '1950-01-01' + (random()*25550)::int)::text AS dob,
      (ARRAY['ON','BC','AB','QC','NS'])[1+(random()*4)::int] AS province
  FROM generate_series(1,100) seq;
""")

print("=== Compliance requirements mapped to PostgreSQL controls ===")
requirements = [
    ("Access control",   "HIPAA §164.312(a), PIPEDA P4.7",
     "RBAC + least privilege + RLS"),
    ("Audit trail",      "HIPAA §164.312(b), SOC 2 CC7",
     "pgaudit + trigger-based audit_log (append-only)"),
    ("Data integrity",   "SOC 2 CC8",
     "FK constraints, NOT NULL, data_checksums=on"),
    ("Transmission sec.","HIPAA §164.312(e)",
     "ssl=on, sslmode=verify-full, pg_hba.conf hostssl"),
    ("Min. necessary",   "HIPAA §164.502(b)",
     "RLS sensitivity filter + de-identified analytics views"),
    ("PHI de-id",        "PIPEDA, GDPR Art.25",
     "Masking views + tokenisation + pgcrypto column encryption"),
]
print(f"  {'Requirement':<18s}  {'Regulation':<28s}  PostgreSQL control")
print("  " + "-"*78)
for req, reg, ctrl in requirements:
    print(f"  {req:<18s}  {reg:<28s}  {ctrl}")

print()
print("pgaudit setup (postgresql.conf):")
print("""
  shared_preload_libraries = 'pgaudit'
  pgaudit.log = 'read,write,ddl,role'
  pgaudit.log_catalog = off
  pgaudit.log_relation = on
  pgaudit.log_parameter = on

  -- Log line format:
  -- AUDIT: SESSION,1,1,READ,SELECT,TABLE,public.patient_records,
  --        SELECT * FROM patient_records WHERE patient_id = $1,<none>
""")


---
## Common Pitfalls

**1. Audit log that can be modified by audited users**
If application roles have UPDATE or DELETE on audit_log, they can cover their tracks. The audit log must be append-only for all application roles: `GRANT INSERT ON audit_log TO app_role` — no UPDATE, no DELETE, ever.

**2. Logging full row content in triggers on large tables**
`row_to_json(OLD)::text` stores the full row for every UPDATE. On wide tables with large text fields this multiplies storage rapidly and slows DML. Log only changed columns, or use pgaudit DDL-level logging for high-throughput tables.

**3. Not verifying audit trigger presence**
Audit triggers can be silently disabled with `ALTER TABLE ... DISABLE TRIGGER`. Regularly verify: `SELECT tgname, tgenabled FROM pg_trigger WHERE tgrelid = 'patient_records'::regclass`. Include trigger presence in automated DQ checks.

**4. Sending PHI to application logs**
`logger.debug(f'Patient: {patient_data}')` may write PHI to log aggregators (Splunk, Datadog) with weaker access controls than the database. Log identifiers only (`patient_id=P001`) — never names, DOBs, or clinical content.

**5. Using production data in development without masking**
Copying production patient records to a dev environment is a HIPAA/PIPEDA violation even if access is restricted. Developers must use masked or synthetically generated data only.

**6. Treating pgaudit as a substitute for application-level audit logs**
pgaudit captures what SQL was executed but not *why* — which user action or business process triggered it. Application-level logs add context: `user=dr.lee action=view_patient patient_id=P001 reason=scheduled_appointment`. Both layers are needed for full compliance audit trails.

---
*sql_methods_library - Samantha McGarrigle*